# Travel Itinerary Planning Concierge

Live Semantic Kernel demo using current flight, hotel, and destination search results. Set `SERPAPI_API_KEY` before running this notebook.


In [ ]:
from datetime import date, timedelta
import os
from travel_concierge.planner_manager import PlannerManager
from travel_concierge.state import AppendOnlyRunLog

if not os.environ.get('SERPAPI_API_KEY'):
    raise RuntimeError('Set SERPAPI_API_KEY before running live searches.')

check_in = date.today() + timedelta(days=60)
check_out = check_in + timedelta(days=3)
request = {
    'origin': 'Boston',
    'origin_airport': 'BOS',
    'destination': 'Kyoto',
    'destination_airport': 'KIX',
    'departure_date': check_in.isoformat(),
    'return_date': check_out.isoformat(),
    'hotel_check_in': check_in.isoformat(),
    'hotel_check_out': check_out.isoformat(),
    'travelers': 2,
    'budget': 2500,
    'interests': ['culture', 'food', 'nature'],
}
result = PlannerManager(AppendOnlyRunLog('run_log.jsonl')).build_itinerary(request)
print('LIVE ITINERARY')
for day in result['days']:
    print(f"\nDay {day['day']}")
    for item in day['items']:
        price = (
            f"{result['currency']} {item['cost']:.2f}"
            if item.get('cost') is not None
            else 'admission price unavailable'
        )
        schedule = item.get('schedule', 'Activity')
        print(f"  - {schedule}: {item['name']} ({price})")
print(f"\nPriced subtotal: {result['currency']} {result['budget']['spent']:.2f}")
print(result['budget_note'])
print(f"Re-plans: {result['replans']}")
print('\nMULTI-AGENT TRACE')
for event in result['trace']:
    details = {key: value for key, value in event.items() if key not in ('timestamp', 'agent', 'action')}
    print(f"{event['agent']}: {event['action']} | {details}")
